In [1]:
import os
import json
import openai
import re
import random
import importlib
import matplotlib.pyplot as plt
from openai import OpenAI

In [2]:
def load_api_key(file_path='api_key.txt'):
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('api_key='):
                return line.strip().split('=', 1)[1]
    return None

api_key = load_api_key()

if api_key is None:
    print("Error: API key not found.")
    exit()

client = OpenAI(api_key=api_key,)

chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ]
)

In [3]:
def get_action_name_array(routine_path):
    
    if not os.path.isfile(routine_path):
        print(f"File not found: {routine_path}")
        return []

    with open(routine_path, 'r', encoding="utf-8") as f:
        routine_content = f.read()

    activity_names = []

    for line in routine_content.splitlines():
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        location_match = re.match(r"^(\d{1,2}:\d{1,2}) - \d{1,2}:\d{1,2}, (.+?): (.+)", line)

        if location_match:
            start_time = location_match.group(1)
            room = location_match.group(2)
            activity = location_match.group(3)
            activity_names.append(activity)

    return activity_names

In [4]:
def separate_blocks(routine_path):
    
    if not os.path.isfile(routine_path):
        print(f"File not found: {routine_path}")
        return []

    with open(routine_path, 'r', encoding="utf-8") as f:
        routine_content = f.read()
    
    routines = []
    current_steps = []
    
    for line in routine_content.splitlines():
        line = line.strip()
        
        if (line == "") and current_steps:
            routines.append(current_steps)
            current_steps = []
        
        else:
            current_steps.append(line)
    
    if current_steps:
        routines.append(current_steps)
        
    return routines

In [5]:
def determine_arubaDict_label(activity_name, routine_block, client):
    max_attempts = 5
    attempt = 0
    arubaDict = ["other", "bed_to_toilet", "eat", "enter_Home", "housekeeping", "leave_home", "meal_preparation", "relax", "resperate", "sleeping", "wash_dishes", "work"]

    system_prompt = (
        f"You are an intelligent assistant helping label smart home activities.\n"
        f"You will be provided with an 'Activity Name' and its corresponding detailed routine steps.\n"
        f"Your task is to choose ONE label from the provided set that best describes the activity.\n"
        f"Return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    # Format the routine block into text
    routine_text = "\n".join(routine_block)

    user_prompt = (
        f"Activity Name: {activity_name}\n\n"
        f"Routine Block:\n{routine_text}\n\n"
        f"Label Set: other, bed_to_toilet, eat, enter_Home, housekeeping, leave_home, meal_preparation, relax, resperate, sleeping, wash_dishes, work\n"
        f"Note: The label 'bed_to_toilet' refers to activities that involve walking from the bed to the bathroom.\n\n"
        f"Which label from the list best fits this activity?\n"
        f"Please return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_label = response.choices[0].message.content.strip().lower().replace(" ", "")

        if selected_label in arubaDict:
            return selected_label
        
        attempt += 1

    return "other"

In [6]:
def determine_milanDict_label(activity_name, routine_block, client):
    max_attempts = 5
    attempt = 0
    milanDict = ["other", "bed_to_toilet", "chores", "desk_Activity", "dining_rm_activity", "evening_meds", "guest_bathroom", "kitchen_activity", "leave_home", "master_bathroom", "master_bedroom_activity", "meditate", "morning_meds", "read", "sleep", "watch_TV"]

    system_prompt = (
        f"You are an intelligent assistant helping label smart home activities.\n"
        f"You will be provided with an 'Activity Name' and its corresponding detailed routine steps.\n"
        f"Your task is to choose ONE label from the provided set that best describes the activity.\n"
        f"Return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    # Format the routine block into text
    routine_text = "\n".join(routine_block)

    user_prompt = (
        f"Activity Name: {activity_name}\n\n"
        f"Routine Block:\n{routine_text}\n\n"
        f"Label Set: other, bed_to_toilet, chores, desk_Activity, dining_rm_activity, evening_meds, guest_bathroom, kitchen_activity, leave_home, master_bathroom, master_bedroom_activity, meditate, morning_meds, read, sleep, watch_TV\n"
        f"Note: The label 'bed_to_toilet' refers to activities that involve walking from the bed to the bathroom.\n\n"
        f"Which label from the list best fits this activity?\n"
        f"Please return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_label = response.choices[0].message.content.strip().lower().replace(" ", "")

        if selected_label in milanDict:
            return selected_label
        
        attempt += 1

    return "other"


In [7]:
def determine_cairoDict_label(activity_name, routine_block, client):
    max_attempts = 5
    attempt = 0
    cairoDict = ["other", "bed_to_toilet", "breakfast", "dinner", "laundry", "leave_home", "lunch", "night_wandering", "work_in_office", "sleep", "take_meds", "wake"]

    system_prompt = (
        f"You are an intelligent assistant helping label smart home activities.\n"
        f"You will be provided with an 'Activity Name' and its corresponding detailed routine steps.\n"
        f"Your task is to choose ONE label from the provided set that best describes the activity.\n"
        f"Return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    # Format the routine block into text
    routine_text = "\n".join(routine_block)

    user_prompt = (
        f"Activity Name: {activity_name}\n\n"
        f"Routine Block:\n{routine_text}\n\n"
        f"Label Set: other, bed_to_toilet, breakfast, dinner, laundry, leave_home, lunch, night_wandering, work_in_office, sleep, take_meds, wake\n"
        f"Note: The label 'bed_to_toilet' refers to activities that involve walking from the bed to the bathroom.\n\n"
        f"Which label from the list best fits this activity?\n"
        f"Please return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_label = response.choices[0].message.content.strip().lower().replace(" ", "")

        if selected_label in cairoDict:
            return selected_label
        
        attempt += 1

    return "other"    

In [8]:
def determine_kyoto_label(activity_name, routine_block, client):
    max_attempts = 5
    attempt = 0
    kyotoDict = ["other", "clean", "meal_preparation", "bed_to_toilet", "personal_hygiene", "sleep", "work", "study", "wash_bathtub", "watch_TV"]
    
    system_prompt = (
        f"You are an intelligent assistant helping label smart home activities.\n"
        f"You will be provided with an 'Activity Name' and its corresponding detailed routine steps.\n"
        f"Your task is to choose ONE label from the provided set that best describes the activity.\n"
        f"Return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )



    # Format the routine block into text
    routine_text = "\n".join(routine_block)

    user_prompt = (
        f"Activity Name: {activity_name}\n\n"
        f"Routine Block:\n{routine_text}\n\n"
        f"Label Set: other, clean, meal_preparation, bed_to_toilet, personal_hygiene, sleep, work, study, wash_bathtub, watch_TV\n"
        f"Note: The label 'bed_to_toilet' refers to activities that involve walking from the bed to the bathroom.\n\n"
        f"Which label from the list best fits this activity?\n"
        f"Please return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )
    

    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_label = response.choices[0].message.content.strip().lower().replace(" ", "")

        if selected_label in kyotoDict:
            return selected_label
        
        attempt += 1

    return "other"

In [9]:
def determine_orange_label(activity_name, routine_block, client):
    max_attempts = 5
    attempt = 0
    orangeDict = ["other", "bathroom|cleaning", "bathroom|showering", "bathroom|using_the_sink", "bathroom|using_the_toilet", "bedroom|cleaning", "bedroom|dressing", "bedroom|napping", "bedroom|reading", "entrance|entering", "entrance|leaving", "kitchen|cleaning", "kitchen|cooking", "kitchen|preparing", "kitchen|washing_the_dishes", "living_room|cleaning", "living_room|computing", "living_room|eating", "living_room|watching_TV", "office|cleaning", "office|computing", "office|watching_TV", "staircase|going_down", "staircase|going_up", "toilet|using_the_toilet"]

    system_prompt = (
        f"You are an intelligent assistant helping label smart home activities.\n"
        f"You will be provided with an 'Activity Name' and its corresponding detailed routine steps.\n"
        f"Your task is to choose ONE label from the provided set that best describes the activity.\n"
        f"Return ONLY the label (exactly the same word appears in the set). Do NOT provide any explanation.\n"
    )


    # Format the routine block into text
    routine_text = "\n".join(routine_block)

    user_prompt = (
        f"Activity Name: {activity_name}\n\n"
        f"Routine Block:\n{routine_text}\n\n"
        f"Label Set: other, bathroom|cleaning, bathroom|showering, bathroom|using_the_sink, bathroom|using_the_toilet, bedroom|cleaning, bedroom|dressing, bedroom|napping, bedroom|reading, entrance|entering, entrance|leaving, kitchen|cleaning, kitchen|cooking, kitchen|preparing, kitchen|washing_the_dishes, living_room|cleaning, living_room|computing, living_room|eating, living_room|watching_TV, office|cleaning, office|computing, office|watching_TV, staircase|going_down, staircase|going_up, toilet|using_the_toilet\n"
        f"Which label from the set best fits this activity?\n"
        f"Return ONLY the label (exactly as it appears in the set). Do NOT provide any explanation.\n"
    )


    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_label = response.choices[0].message.content.strip().lower().replace(" ", "")

        if selected_label in orangeDict:
            return selected_label
        
        attempt += 1

    return "other"


In [19]:
def process_one_pair(regenerate_file, original_file, output_folder, client):
    # Make sure output folder exists
    os.makedirs(output_folder, exist_ok=True)
        
    if not os.path.exists(original_file):
        print(f"🚨 Original file not found: {original_file}")
        return

    if not os.path.exists(regenerate_file):
        print(f"🚨 Regenerated file not found: {regenerate_file}")
        return    
    
    print(f"[AgentSense] Processing pair:")
    print(f"[AgentSense] Step_4_file: {original_file}")
    print(f"[AgentSense] Step_7_file: {regenerate_file}")

    # Load names and blocks
    activity_names = get_action_name_array(original_file)
    routines = separate_blocks(regenerate_file)

    # Sanity check
    if len(activity_names) != len(routines):
        print(f"Skipping {filename}: {len(activity_names)} names vs {len(routines)} blocks")
        return

    # Prepare output file
    regenerate_filename = os.path.basename(regenerate_file)
    output_filename = f"label_{regenerate_filename}"
    output_pathfile = os.path.join(output_folder, output_filename)

    with open(output_pathfile, "w") as out_f:
        # For each block
        for act_name, block in zip(activity_names, routines):
            # compute the three labels
            aruba_label = determine_arubaDict_label(act_name, block, client)
            milan_label = determine_milanDict_label(act_name, block, client)
            cairo_label = determine_cairoDict_label(act_name, block, client)
            kyoto_label = determine_kyoto_label(act_name, block, client)
            orange_label = determine_orange_label(act_name, block, client)

            # write each action line with labels appended
            for action in block:
                out_f.write(f"{action} ({aruba_label}, {milan_label}, {cairo_label}, {kyoto_label}, {orange_label})\n")

            # optional blank line between blocks
            out_f.write("\n")

    print(f"Wrote labeled file: {output_filename}")
    


In [22]:
# =========================
# Configuration (User-editable)
# =========================
step_7_file = "data/step_7_data/regenerated_Sarah_routine_env_0_Sunday_parsed_grounded.txt"
step_4_file = "data/step_4_data/Sarah_routine_env_0_Sunday.txt"
output_folder = "data/step_8_data"

# IMPORTANT:
# The step_7 file and step_4 file MUST correspond to the same
# synthetic character, environment, and day.

process_one_pair(step_7_file, step_4_file, output_folder, client)

[AgentSense] Processing pair:
[AgentSense] Step_4_file: data/step_4_data/Sarah_routine_env_0_Sunday.txt
[AgentSense] Step_7_file: data/step_7_data/regenerated_Sarah_routine_env_0_Sunday_parsed_grounded.txt
Wrote labeled file: label_regenerated_Sarah_routine_env_0_Sunday_parsed_grounded.txt
